In [1]:
import pandas as pd
import pickle

In [2]:
test = pd.read_csv("test.csv")

In [3]:
test_data = test
# datetime列をdatetime64型に変換
test_data["datetime"] = pd.to_datetime(test_data["datetime"])

# 日付の範囲を決定
min_date = test_data["datetime"].min()
max_date = test_data["datetime"].max()

# 日付を0から1の範囲にスケーリング
test_data["normalized_datetime"] = (test_data["datetime"] - min_date) / (max_date - min_date)

test_data = test_data.drop(columns=["datetime"])

In [4]:
# price_amとprice_pmの値をすべて1増やす
test_data.loc[:, ["price_am"]] += 1

In [5]:
# closeが1の行のamとpmを0に変更
test_data.loc[test_data["close"] == 1, "price_am"] = 0
test_data.loc[test_data["close"] == 1, "price_pm"] = 0

In [6]:
test_price = test_data.pop("price_pm")
test_data.insert(test_data.columns.get_loc("normalized_datetime") + 1, "price_pm", test_price)

In [7]:
test_data.head()

,client,close,price_am,normalized_datetime,price_pm
0,1,0,4,0.000000,2
1,0,0,6,0.002747,5
2,1,0,3,0.005495,2
3,1,0,2,0.008242,1
4,0,0,2,0.010989,1


In [8]:
#モデルの読み込み
with open("Apple_model.pickle", "rb") as fp:
    model = pickle.load(fp)

In [9]:
predict_y = model.predict(test_data)

In [10]:
test["y"] = predict_y

In [11]:
inference = test.drop(columns=["client", "close", "price_am", "price_pm", "normalized_datetime"])

In [12]:
inference.head()

,datetime,y
0,2016-04-01,30.985
1,2016-04-02,33.975
2,2016-04-03,30.985
3,2016-04-04,24.765
4,2016-04-05,24.765


In [13]:
# 'datetime'列を日付型に変換
#inference['datetime'] = pd.to_datetime(inference['datetime'], format='%Y%m%d')
# datetime列を指定された形式に変換
#inference['datetime'] = inference['datetime'].dt.strftime('%Y-%m-%d')

In [14]:
# 'y'列のデータ型をfloatに変換
#inference['y'] = inference['y'].astype(float)

In [15]:
#inference.head()

In [16]:
inference.to_csv("inference.csv", index=False, header=None)

In [17]:
inference.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 365 entries, 0 to 364
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   datetime  365 non-null    datetime64[ns]
 1   y         365 non-null    float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 5.8 KB
